# TP 2 : corrigé exécutable

**Prérequis** (comme pour les étudiants) : TP 1 Databricks terminé, ou notebook `tp2-00-rattrapage-databricks.ipynb` exécuté. Il faut le volume `workspace.shopflow.raw` (fichiers SF1 dans `raw/sf1/<table>/part-AAAA-MM.parquet`) et les tables `orders`, `customers`, `payments` dans `workspace.shopflow`.

**Avant de lancer :** attacher le notebook au calcul **Serverless**, laisser `RESET = True` (repart d'un état propre : nécessaire pour que les comptages et la version « avant rétrofacturations 2025 » soient justes), puis *Run all*. Durée : surtout la création des fichiers (partie 0) et le `MERGE`.

## Paramètres et remise à zéro

In [0]:
import os, glob, time
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

CATALOG, SCHEMA = "workspace", "shopflow"
RAW  = "/Volumes/workspace/shopflow/raw/sf1"
LAND = "/Volumes/workspace/shopflow/landing"
CHK  = "/Volumes/workspace/shopflow/checkpoints"

RESET = False

spark.sql("USE workspace.shopflow")
spark.conf.set("spark.sql.session.timeZone", "UTC")   # comme au TP 1

In [0]:
# Vérification des prérequis : 24 fichiers mensuels de paiements et les tables du TP 1
nb_fichiers = len(glob.glob(f"{RAW}/payments/part-*.parquet"))
print("fichiers payments dans raw :", nb_fichiers, "(attendu : 24)")
for t in ["orders", "customers", "payments"]:
    print(t, spark.table(f"workspace.shopflow.{t}").count())
assert nb_fichiers == 24, "Lancer d'abord le TP 1 Databricks ou tp2-00-rattrapage-databricks.ipynb"

fichiers payments dans raw : 24 (attendu : 24)
orders 1011946
customers 50001
payments 1031980


In [0]:
TP2_TABLES = ["bronze_payments", "bronze_chargebacks", "bronze_clickstream",
              "silver_payments", "silver_payments_quarantine",
              "gold_payment_features", "payments_fragmented"]

if RESET:
    for t in TP2_TABLES:
        spark.sql(f"DROP TABLE IF EXISTS workspace.shopflow.{t}")
    for base in [LAND, CHK]:
        if os.path.exists(base):
            for d in os.listdir(base):
                dbutils.fs.rm(f"{base}/{d}", True)
    print("Remise à zéro faite")

---
## Partie 0 : simuler l'arrivée des fichiers

Code fourni aux étudiants, recopié sans modification (sauf les imports, déjà faits plus haut).

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.shopflow.landing;
CREATE VOLUME IF NOT EXISTS workspace.shopflow.checkpoints;

In [0]:
for d in ["payments", "chargebacks", "clickstream"]:
    os.makedirs(f"{LAND}/{d}", exist_ok=True)

HIDDEN = ["is_fraud", "fraud_scenario", "chargeback_ts"]

def land_payments(year: int) -> None:
    """Un fichier CSV par mois, avec doublons et lignes invalides."""
    for f in sorted(glob.glob(f"{RAW}/payments/part-{year}-*.parquet")):
        month = f.split("part-")[1][:7]
        df = pd.read_parquet(f).drop(columns=HIDDEN)
        dup = df.sample(frac=0.005, random_state=1)                      # renvois en double
        neg = df.sample(frac=0.001, random_state=2).assign(              # montants négatifs
            amount=lambda d: -d.amount, payment_id=lambda d: d.payment_id + 10_000_000)
        noid = df.sample(frac=0.0005, random_state=3).assign(payment_id=pd.NA)  # identifiant manquant
        out = pd.concat([df, dup, neg, noid]).sample(frac=1, random_state=4)
        out.to_csv(f"{LAND}/payments/payments_{month}.csv", index=False)
        print("écrit", month, len(out))

def land_chargebacks(start: str, end: str, name: str) -> None:
    """Rétrofacturations reçues entre start (inclus) et end (exclu)."""
    cols = ["payment_id", "chargeback_ts"]
    cb = pd.concat(pd.read_parquet(f, columns=cols) for f in glob.glob(f"{RAW}/payments/*.parquet"))
    cb = cb.dropna(subset=["chargeback_ts"])
    lo, hi = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    cb = cb[(cb.chargeback_ts >= lo) & (cb.chargeback_ts < hi)].assign(reason="fraud")
    cb.to_csv(f"{LAND}/chargebacks/chargebacks_{name}.csv", index=False)
    print("écrit", name, len(cb))

def land_clicks(month: str) -> None:
    """Un mois de navigation en JSON lines, avec 100 lignes corrompues."""
    ev = pd.read_parquet(f"{RAW}/clickstream/part-{month}.parquet")
    path = f"{LAND}/clickstream/clicks_{month}.json"
    # Un volume n'accepte pas l'ajout en fin de fichier (mode "a") :
    # on prépare tout le texte en mémoire, puis on l'écrit en une seule fois.
    texte = ev.to_json(orient="records", lines=True, date_format="iso").rstrip("\n") + "\n"
    texte += '{"event_id": "abc", "event_ts": "pas-une-date", "event_type": \n' * 100
    with open(path, "w") as fh:
        fh.write(texte)
    print("écrit", month, len(ev))

In [0]:
# Premier lot : l'année 2024
land_payments(2024)
land_chargebacks("2024-01-01", "2025-01-01", "2024")
land_clicks("2024-12")

écrit 2024-01 34707
écrit 2024-02 29716
écrit 2024-03 33902
écrit 2024-04 34387
écrit 2024-05 35738
écrit 2024-06 37373
écrit 2024-07 33886
écrit 2024-08 30370
écrit 2024-09 37723
écrit 2024-10 41621
écrit 2024-11 61130
écrit 2024-12 68446
écrit 2024 2671
écrit 2024-12 1053458


In [0]:
# Étape 3 : 12 fichiers de paiements, 1 de rétrofacturations, 1 de navigation
for d in ["payments", "chargebacks", "clickstream"]:
    fichiers = sorted(os.listdir(f"{LAND}/{d}"))
    print(f"{d:12s} {len(fichiers):3d} fichier(s)  ex. {fichiers[0]}")

payments      12 fichier(s)  ex. payments_2024-01.csv
chargebacks    1 fichier(s)  ex. chargebacks_2024.csv
clickstream    1 fichier(s)  ex. clicks_2024-12.json


**Q0.** Au moment du paiement, l'entreprise ne sait pas si c'est une fraude. `is_fraud` est une vérité terrain que seul un jeu synthétique possède. Dans la réalité, la seule information est la **rétrofacturation** : la banque du porteur conteste le paiement des semaines plus tard (7 à 60 jours dans notre générateur), et environ 10 % des fraudes ne sont jamais signalées. Garder `is_fraud` dans les fichiers serait une **fuite d'information** (*label leakage*).

---
## Partie 1 : bronze, ingestion incrémentale

In [0]:
def ingest(source: str, fmt: str) -> None:
    """Ingère les NOUVEAUX fichiers de landing/<source> dans bronze_<source>, puis s'arrête."""
    stream = (spark.readStream.format("cloudFiles")
              .option("cloudFiles.format", fmt)
              .option("cloudFiles.schemaLocation", f"{CHK}/{source}/schema")
              .option("header", "true")
              .load(f"{LAND}/{source}")
              .withColumn("_ingested_at", F.current_timestamp())
              .withColumn("_source_file", F.col("_metadata.file_path")))
    query = (stream.writeStream
             .option("checkpointLocation", f"{CHK}/{source}/checkpoint")
             .trigger(availableNow=True)          # seul mode accepté sur le calcul serverless
             .toTable(f"workspace.shopflow.bronze_{source}"))
    query.awaitTermination()

def ingest_all() -> None:
    for source, fmt in [("payments", "csv"), ("chargebacks", "csv"), ("clickstream", "json")]:
        ingest(source, fmt)
        print(f"bronze_{source} :", spark.table(f"workspace.shopflow.bronze_{source}").count(), "lignes")

In [0]:
# Étape 2 : ingestion des trois sources
ingest_all()

bronze_payments : 478999 lignes
bronze_chargebacks : 2671 lignes
bronze_clickstream : 1053558 lignes


Étape 3 : schéma de `bronze_payments` (réponse à Q1 : tout est en `STRING`).

In [0]:
%sql
DESCRIBE TABLE workspace.shopflow.bronze_payments;

col_name,data_type,comment
payment_id,string,null
order_id,string,null
customer_id,string,null
card_id,string,null
payment_ts,string,null
amount,string,null
currency,string,null
payment_method,string,null
card_country,string,null
ip,string,null


In [0]:
%sql
SELECT _source_file, COUNT(*) AS lignes
FROM workspace.shopflow.bronze_payments
GROUP BY _source_file ORDER BY _source_file;

_source_file,lignes
/Volumes/workspace/shopflow/landing/payments/payments_2024-01.csv,34707
/Volumes/workspace/shopflow/landing/payments/payments_2024-02.csv,29716
/Volumes/workspace/shopflow/landing/payments/payments_2024-03.csv,33902
/Volumes/workspace/shopflow/landing/payments/payments_2024-04.csv,34387
/Volumes/workspace/shopflow/landing/payments/payments_2024-05.csv,35738
/Volumes/workspace/shopflow/landing/payments/payments_2024-06.csv,37373
/Volumes/workspace/shopflow/landing/payments/payments_2024-07.csv,33886
/Volumes/workspace/shopflow/landing/payments/payments_2024-08.csv,30370
/Volumes/workspace/shopflow/landing/payments/payments_2024-09.csv,37723
/Volumes/workspace/shopflow/landing/payments/payments_2024-10.csv,41621


In [0]:
# Étape 4 : relance de l'ingestion des paiements (réponse à Q2 : rien n'est ajouté)
avant = spark.table("workspace.shopflow.bronze_payments").count()
ingest("payments", "csv")
apres = spark.table("workspace.shopflow.bronze_payments").count()
print(f"avant : {avant}  après : {apres}  ajoutées : {apres - avant}  (attendu : 0)")

avant : 478999  après : 478999  ajoutées : 0  (attendu : 0)


In [0]:
%sql
SELECT _source_file, event_id, event_ts, event_type, _rescued_data
FROM workspace.shopflow.bronze_clickstream
WHERE event_type IS NULL
LIMIT 20;

_source_file,event_id,event_ts,event_type,_rescued_data
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null
/Volumes/workspace/shopflow/landing/clickstream/clicks_2024-12.json,null,null,null,null


In [0]:
%sql
SELECT COUNT(*) AS lignes_sans_event_type,
       COUNT_IF(_rescued_data IS NOT NULL) AS dont_rescued_data_rempli
FROM workspace.shopflow.bronze_clickstream
WHERE event_type IS NULL;

lignes_sans_event_type,dont_rescued_data_rempli
100,0


**Q1.** Toutes les colonnes sont en `STRING` (plus `_rescued_data`, `_ingested_at`, `_source_file`) : Auto Loader infère tout en texte pour CSV et JSON, sauf si `cloudFiles.inferColumnTypes` vaut `true`. C'est un bon choix pour le bronze : aucune ligne n'est perdue à cause d'un type inattendu ; le typage se fait en silver, où l'on décide quoi faire des valeurs invalides.

**Q2.** Rien n'est ajouté. Auto Loader enregistre dans le **point de reprise** (`checkpointLocation`) la liste des fichiers déjà ingérés et ne traite que les nouveaux. Avec `trigger(availableNow=True)`, la requête traite tout ce qui est nouveau puis s'arrête.

**Q3.** En mode `PERMISSIVE` (par défaut), les lignes JSON illisibles donnent probablement des lignes dont les colonnes sont nulles ; `_rescued_data` capture plutôt les colonnes inattendues ou les incompatibilités de type. **Noter ici ce que montrent les deux requêtes ci-dessus** (nombre attendu : 100 lignes ). L'option `badRecordsPath` permet d'écrire les enregistrements malformés dans des fichiers à part.

---
## Partie 2 : silver, qualité et `MERGE`

### 2.1 Créer la table silver

L'énoncé enchaîne `CREATE TABLE IF NOT EXISTS` et `ADD CONSTRAINT` : relancé, `ADD CONSTRAINT` échoue car la contrainte existe déjà (piège voulu, discussion sur l'idempotence du DDL). Ici, la contrainte n'est ajoutée que si elle est absente, pour que *Run all* passe.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.shopflow.silver_payments (
  payment_id BIGINT NOT NULL,
  order_id BIGINT,
  customer_id BIGINT,
  card_id BIGINT,
  payment_ts TIMESTAMP NOT NULL,
  amount DOUBLE,
  currency STRING,
  payment_method STRING,
  card_country STRING,
  ip STRING,
  status STRING,
  has_chargeback BOOLEAN,
  chargeback_ts TIMESTAMP
) CLUSTER BY (payment_ts);

In [0]:
props = {r["key"] for r in spark.sql("SHOW TBLPROPERTIES workspace.shopflow.silver_payments").collect()}
if "delta.constraints.amount_positive" not in props:
    spark.sql("ALTER TABLE workspace.shopflow.silver_payments ADD CONSTRAINT amount_positive CHECK (amount >= 0)")
    print("contrainte ajoutée")
else:
    print("contrainte déjà présente")

contrainte ajoutée


### 2.2 à 2.6 : le traitement silver

Les étapes 2.2 (typage), 2.3 (quarantaine), 2.4 (dédoublonnage + `MERGE`) et 2.6 (rétrofacturations) sont regroupées dans une fonction `silver_pipeline()`, car la partie 3 demande de **relancer tout le traitement silver** après le deuxième lot. La fonction renvoie la version de `silver_payments` juste avant le `MERGE` des rétrofacturations (utile pour la partie 3).

In [0]:
SQL_TYPED = """
CREATE OR REPLACE TEMP VIEW payments_typed AS
SELECT
  try_cast(payment_id AS BIGINT)     AS payment_id,
  try_cast(order_id AS BIGINT)       AS order_id,
  try_cast(customer_id AS BIGINT)    AS customer_id,
  try_cast(card_id AS BIGINT)        AS card_id,
  try_cast(payment_ts AS TIMESTAMP)  AS payment_ts,
  try_cast(amount AS DOUBLE)         AS amount,
  currency, payment_method, card_country, ip, status,
  _ingested_at, _source_file
FROM workspace.shopflow.bronze_payments
"""

# Recréée à chaque exécution, donc idempotente
SQL_QUARANTINE = """
CREATE OR REPLACE TABLE workspace.shopflow.silver_payments_quarantine AS
SELECT *,
  CASE WHEN payment_id IS NULL THEN 'payment_id manquant'
       WHEN payment_ts IS NULL THEN 'payment_ts invalide'
       ELSE 'montant négatif' END AS reject_reason
FROM payments_typed
WHERE payment_id IS NULL OR payment_ts IS NULL OR amount < 0
"""

SQL_MERGE_PAYMENTS = """
MERGE INTO workspace.shopflow.silver_payments AS t
USING (
  SELECT * EXCEPT (rn, _ingested_at, _source_file)
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY _ingested_at) AS rn
    FROM payments_typed
    WHERE payment_id IS NOT NULL AND payment_ts IS NOT NULL AND amount >= 0
  )
  WHERE rn = 1
) AS s
ON t.payment_id = s.payment_id
WHEN NOT MATCHED THEN INSERT (
  payment_id, order_id, customer_id, card_id, payment_ts, amount, currency,
  payment_method, card_country, ip, status, has_chargeback, chargeback_ts)
VALUES (
  s.payment_id, s.order_id, s.customer_id, s.card_id, s.payment_ts, s.amount, s.currency,
  s.payment_method, s.card_country, s.ip, s.status, false, NULL)
"""

SQL_MERGE_CHARGEBACKS = """
MERGE INTO workspace.shopflow.silver_payments AS t
USING (
  SELECT try_cast(payment_id AS BIGINT) AS payment_id,
         MIN(try_cast(chargeback_ts AS TIMESTAMP)) AS chargeback_ts
  FROM workspace.shopflow.bronze_chargebacks
  GROUP BY 1
) AS s
ON t.payment_id = s.payment_id
WHEN MATCHED AND NOT t.has_chargeback THEN
  UPDATE SET has_chargeback = true, chargeback_ts = s.chargeback_ts
"""

def current_version(table: str = "workspace.shopflow.silver_payments") -> int:
    return spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]

def silver_pipeline() -> int:
    spark.sql(SQL_TYPED)                                         # 2.2
    spark.sql(SQL_QUARANTINE)                                    # 2.3
    print("MERGE paiements       :", spark.sql(SQL_MERGE_PAYMENTS).first().asDict())     # 2.4
    v_before_cb = current_version()
    print("MERGE rétrofacturations:", spark.sql(SQL_MERGE_CHARGEBACKS).first().asDict())  # 2.6
    print("silver_payments        :", spark.table("workspace.shopflow.silver_payments").count())
    print("quarantaine            :", spark.table("workspace.shopflow.silver_payments_quarantine").count())
    return v_before_cb

In [0]:
v_avant_cb_lot1 = silver_pipeline()

MERGE paiements       : {'num_affected_rows': 475904, 'num_updated_rows': 0, 'num_deleted_rows': 0, 'num_inserted_rows': 475904}
MERGE rétrofacturations: {'num_affected_rows': 2671, 'num_updated_rows': 2671, 'num_deleted_rows': 0, 'num_inserted_rows': 0}
silver_payments        : 475904
quarantaine            : 715


### Points de contrôle

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.shopflow.silver_payments)                               AS silver,
  (SELECT COUNT(*) FROM workspace.shopflow.payments WHERE year(payment_ts) = 2024)        AS tp1_2024,
  (SELECT COUNT(*) FROM workspace.shopflow.silver_payments_quarantine)                    AS quarantaine,
  (SELECT COUNT(*) FROM workspace.shopflow.bronze_payments)                               AS recues,
  ROUND(100 * quarantaine / recues, 3)                                                    AS pct_quarantaine;  -- attendu ≈ 0,15 %

silver,tp1_2024,quarantaine,recues,pct_quarantaine
475904,475902,715,478999,0.149


In [0]:
%sql
SELECT reject_reason, COUNT(*) AS lignes
FROM workspace.shopflow.silver_payments_quarantine
GROUP BY reject_reason;

reject_reason,lignes
montant négatif,476
payment_id manquant,239


### 2.5 Tester

Insertion d'un montant négatif directement dans `silver_payments` : elle doit être refusée par la contrainte `amount_positive` (l'erreur est capturée pour ne pas arrêter *Run all*).

In [0]:
try:
    spark.sql("""
      INSERT INTO workspace.shopflow.silver_payments VALUES
      (999999999, 1, 1, 11, TIMESTAMP'2024-06-01 10:00:00', -10.0, 'EUR', 'card', 'FR',
       '10.0.0.1', 'authorized', false, NULL)
    """)
    print("⚠️ insertion acceptée : la contrainte ne fonctionne pas")
except Exception as e:
    print("Refusé, comme attendu :", str(e).splitlines()[0][:300])

Refusé, comme attendu : (com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException) [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint amount_positive (amount >= 0) violated by row with values:


In [0]:
# Relance complète du traitement silver : le nombre de lignes ne doit pas changer
n_avant = spark.table("workspace.shopflow.silver_payments").count()
silver_pipeline()
n_apres = spark.table("workspace.shopflow.silver_payments").count()
print(f"avant : {n_avant}  après : {n_apres}  (attendu : identique)")
v_fin_lot1 = current_version()   # état de la table à la fin du lot 1 (sert en partie 3, Q7)

MERGE paiements       : {'num_affected_rows': 0, 'num_updated_rows': 0, 'num_deleted_rows': 0, 'num_inserted_rows': 0}
MERGE rétrofacturations: {'num_affected_rows': 0, 'num_updated_rows': 0, 'num_deleted_rows': 0, 'num_inserted_rows': 0}
silver_payments        : 475904
quarantaine            : 715
avant : 475904  après : 475904  (attendu : identique)


**Q4.** Un `INSERT` relancé insère une seconde fois les mêmes lignes. Un `MERGE` sur la clé n'insère que les clés absentes : relancer le traitement donne le même résultat, il est **idempotent**. C'est indispensable car un pipeline est relancé après chaque panne.

**Q5.** La contrainte fait échouer **toute** l'écriture pour une seule ligne fautive, ce qui bloque le pipeline. La quarantaine laisse passer les bonnes lignes et garde les mauvaises pour analyse. Les deux se complètent : la quarantaine filtre, la contrainte garantit qu'aucune erreur de code ne laisse passer une ligne invalide.

Points de vigilance : les lignes à montant négatif ont un `payment_id` décalé de 10 000 000, ce ne sont pas des doublons, elles finissent en quarantaine. Taux attendu : (0,1 % + 0,05 %) / 1,0065 ≈ 0,15 % des lignes reçues.

### 2.6 Contrôle des rétrofacturations (lot 1)

In [0]:
%sql
SELECT COUNT_IF(has_chargeback) AS paiements_retrofactures,
       MIN(chargeback_ts) AS premiere, MAX(chargeback_ts) AS derniere
FROM workspace.shopflow.silver_payments;

paiements_retrofactures,premiere,derniere
2671,2024-01-12T21:58:37.000Z,2024-12-31T23:04:52.000Z


---
## Partie 3 : deuxième lot et *time travel*

In [0]:
# Étape 1 : deuxième lot
land_payments(2025)
land_chargebacks("2025-01-01", "2026-04-01", "2025")

écrit 2025-01 41131
écrit 2025-02 35027
écrit 2025-03 39962
écrit 2025-04 40504
écrit 2025-05 41916
écrit 2025-06 43780
écrit 2025-07 39710
écrit 2025-08 35452
écrit 2025-09 43917
écrit 2025-10 48336
écrit 2025-11 70831
écrit 2025-12 79123
écrit 2025 4320


In [0]:
# Étape 2 : bronze (seuls les nouveaux fichiers sont lus) puis silver en entier
ingest_all()
v_avant_cb_2025 = silver_pipeline()
print("Version juste avant le MERGE des rétrofacturations 2025 :", v_avant_cb_2025)

bronze_payments : 1038688 lignes
bronze_chargebacks : 6991 lignes
bronze_clickstream : 1053558 lignes
MERGE paiements       : {'num_affected_rows': 556076, 'num_updated_rows': 0, 'num_deleted_rows': 0, 'num_inserted_rows': 556076}
MERGE rétrofacturations: {'num_affected_rows': 4320, 'num_updated_rows': 4320, 'num_deleted_rows': 0, 'num_inserted_rows': 0}
silver_payments        : 1031980
quarantaine            : 1549
Version juste avant le MERGE des rétrofacturations 2025 : 6


Étape 3 : historique de la table.

In [0]:
%sql
SELECT version, timestamp, operation, operationMetrics
FROM (DESCRIBE HISTORY workspace.shopflow.silver_payments)
ORDER BY version;

version,timestamp,operation,operationMetrics
0,2026-11-03T15:56:33.000Z,CREATE TABLE,Map()
1,2026-11-03T15:56:36.000Z,ADD CONSTRAINT,Map()
2,2026-11-03T15:56:47.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 6954802, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3167, materializeSourceTimeMs -> 22, numTargetRowsInserted -> 475904, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 475904, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 475904, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3079)"
3,2026-11-03T15:56:55.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 76853, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2671, executionTimeMs -> 5151, materializeSourceTimeMs -> 867, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1679, numTargetRowsUpdated -> 2671, numOutputRows -> 2671, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2671, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2542)"
4,2026-11-03T15:57:11.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2042, materializeSourceTimeMs -> 12, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 475904, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1970)"
5,2026-11-03T15:57:19.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1569, materializeSourceTimeMs -> 450, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1055, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2671, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 0)"
6,2026-11-03T15:58:12.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 8642412, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3292, materializeSourceTimeMs -> 13, numTargetRowsInserted -> 556076, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 556076, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1031980, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3244)"
7,2026-11-03T15:58:18.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 128938, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 2, numTargetRowsMatchedUpdated -> 4320, executionTimeMs -> 4306, materializeSourceTimeMs -> 524, numTargetRowsInser

Étape 4 : la version cherchée est celle qui précède le dernier `MERGE` dont `operationMetrics.numTargetRowsUpdated` est non nul. La fonction `silver_pipeline()` l'a mémorisée dans `v_avant_cb_2025`.

In [0]:
q = """
SELECT COUNT_IF(has_chargeback) AS fraudes_connues
FROM workspace.shopflow.silver_payments {version}
WHERE payment_ts >= '2024-12-01' AND payment_ts < '2025-01-01'
"""
avant = spark.sql(q.format(version=f"VERSION AS OF {v_avant_cb_2025}")).first()[0]
maintenant = spark.sql(q.format(version="")).first()[0]
print(f"Fraudes connues pour décembre 2024 : version {v_avant_cb_2025} = {avant}, version courante = {maintenant}")

Fraudes connues pour décembre 2024 : version 6 = 78, version courante = 510


Démonstration pour Q7 : `TIMESTAMP AS OF` avec l'heure de fin du lot 1 (dans la vraie vie, on écrirait `TIMESTAMP AS OF '2025-01-01'` ; ici la table a été créée aujourd'hui, donc on prend l'heure de la dernière version du lot 1, mémorisée dans `v_fin_lot1`). On doit retrouver uniquement les paiements 2024.

In [0]:
hist = spark.sql("DESCRIBE HISTORY workspace.shopflow.silver_payments").toPandas()
ts_lot1 = hist.loc[hist.version == v_fin_lot1, "timestamp"].iloc[0]
print("Données vues par un modèle entraîné à", ts_lot1)
display(spark.sql(f"""
  SELECT year(payment_ts) AS annee, COUNT(*) AS paiements, COUNT_IF(has_chargeback) AS fraudes_connues
  FROM workspace.shopflow.silver_payments TIMESTAMP AS OF '{ts_lot1}'
  GROUP BY 1 ORDER BY 1
"""))

Données vues par un modèle entraîné à 2026-11-03 15:57:19


annee,paiements,fraudes_connues
2024,475902,2671
2025,2,0


**Q6.** Les rétrofacturations arrivent 7 à 60 jours après le paiement. Celles des paiements de décembre 2024 reçues en janvier ou février 2025 sont dans le fichier du lot 2 : avant ce `MERGE`, ces fraudes étaient vues comme légitimes.

**Q7.** `SELECT ... FROM silver_payments TIMESTAMP AS OF '2025-01-01'` (ou `VERSION AS OF`). C'est essentiel pour **reproduire** un entraînement, auditer une décision, et comprendre que les labels récents sont incomplets. Le *Fraud Detection Handbook* recommande un « délai » entre la période d'entraînement et la période de test pour cette raison (chap. 5). Attention : après un `VACUUM`, les versions anciennes ne sont plus lisibles.

---
## Partie 4 : gold, la table de features

Une ligne par paiement de `silver_payments`. Toutes les fenêtres s'arrêtent à la ligne courante (`PRECEDING ... CURRENT ROW`) : aucune feature n'utilise le futur.

In [0]:
%sql
USE workspace.shopflow;

CREATE OR REPLACE TABLE gold_payment_features CLUSTER BY (payment_ts) AS
WITH base AS (
  SELECT
    p.payment_id, p.payment_ts, p.amount, p.status, p.ip, p.customer_id,
    coalesce(p.card_id, -p.payment_id) AS card_key,      -- PayPal : une clé unique par paiement
    unix_timestamp(p.payment_ts) AS ts_s,
    o.shipping_country,
    c.country AS customer_country,
    c.signup_date,
    p.has_chargeback
  FROM silver_payments AS p
  LEFT JOIN orders AS o ON o.order_id = p.order_id
  LEFT JOIN customers AS c ON c.customer_id = p.customer_id
)
SELECT
  payment_id, payment_ts, amount, status,
  COUNT(*) OVER w_card_1d AS card_nb_tx_1d,
  COUNT(*) OVER w_card_7d AS card_nb_tx_7d,
  AVG(amount) OVER w_card_7d AS card_avg_amount_7d,
  AVG(amount) OVER w_card_30d AS card_avg_amount_30d,
  amount / AVG(amount) OVER w_card_30d AS amount_ratio_30d,
  COUNT(*) OVER w_ip_1h AS ip_nb_tx_1h,
  size(collect_set(card_key) OVER w_ip_1h) AS ip_nb_cards_1h,
  -- si collect_set en fenêtre RANGE est refusé : approx_count_distinct(card_key) OVER w_ip_1h
  CASE WHEN customer_id = 0 THEN -1
       ELSE datediff(to_date(payment_ts), signup_date) END AS account_age_days,
  coalesce(shipping_country <> customer_country, false) AS ship_country_mismatch,
  customer_id = 0 AS is_guest,
  has_chargeback AS label
FROM base
WINDOW
  w_card_1d  AS (PARTITION BY card_key ORDER BY ts_s RANGE BETWEEN 86400 PRECEDING AND CURRENT ROW),
  w_card_7d  AS (PARTITION BY card_key ORDER BY ts_s RANGE BETWEEN 604800 PRECEDING AND CURRENT ROW),
  w_card_30d AS (PARTITION BY card_key ORDER BY ts_s RANGE BETWEEN 2592000 PRECEDING AND CURRENT ROW),
  w_ip_1h    AS (PARTITION BY ip ORDER BY ts_s RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW);

num_affected_rows,num_inserted_rows


**Point de contrôle** : autant de lignes que `silver_payments`.

In [0]:
%sql
SELECT (SELECT COUNT(*) FROM workspace.shopflow.gold_payment_features) AS gold,
       (SELECT COUNT(*) FROM workspace.shopflow.silver_payments)       AS silver;

gold,silver
1031980,1031980


Vérification de bon sens (requête de l'énoncé).

In [0]:
%sql
SELECT label, COUNT(*) AS paiements,
       ROUND(AVG(ip_nb_cards_1h), 2) AS cartes_par_ip, ROUND(AVG(amount_ratio_30d), 2) AS ratio,
       ROUND(AVG(CAST(ship_country_mismatch AS INT)), 3) AS mismatch
FROM workspace.shopflow.gold_payment_features
GROUP BY label;

label,paiements,cartes_par_ip,ratio,mismatch
false,1024989,1.06,1.0,0.001
true,6991,2.85,1.24,0.854


**Q8**  :
- les moyennes de `ip_nb_cards_1h` , `amount_ratio_30d` , `ship_country_mismatch` sont plus pour les valeurs à true 
- `ship_country_mismatch` vrai pour `compte_compromis` et `nouveau_compte_montant` (livraison vers un pays « à risque »).
- `amount_ratio_30d` élevé pour `compte_compromis` (achats high-tech sur un compte habituellement modeste).
- `account_age_days` faible (0 à 2 jours) pour `nouveau_compte_montant`.

**Q9.** En production, au moment de décider, les paiements futurs n'existent pas. Une feature qui utilise le futur donne d'excellents résultats hors ligne puis s'effondre en production : c'est une fuite temporelle. Les fenêtres `PRECEDING` garantissent qu'on calcule hors ligne exactement ce qu'on pourra calculer en ligne (séance 5). Le paiement courant est inclus (`CURRENT ROW`) : c'est acceptable, son montant est connu au moment de la décision.

---
## Partie 5 : maintenance

In [0]:
# Étape 1 : table volontairement fragmentée
(spark.table("workspace.shopflow.silver_payments")
      .repartition(400)
      .write.mode("overwrite").saveAsTable("workspace.shopflow.payments_fragmented"))

In [0]:
def mesure(etiquette: str) -> None:
    d = spark.sql("DESCRIBE DETAIL workspace.shopflow.payments_fragmented").first()
    t0 = time.perf_counter()
    spark.sql("""SELECT card_country, SUM(amount) FROM workspace.shopflow.payments_fragmented
                 GROUP BY card_country""").collect()
    duree = time.perf_counter() - t0
    print(f"{etiquette:6s} numFiles={d['numFiles']:4d}  sizeInBytes={d['sizeInBytes']:,}  requête={duree:.2f} s")

# Étape 2
mesure("avant")

avant  numFiles= 400  sizeInBytes=24,506,843  requête=2.76 s


In [0]:
%sql
-- Étape 3
OPTIMIZE workspace.shopflow.payments_fragmented;

path,metrics
,"List(1, 400, List(20352209, 20352209, 2.0352209E7, 1, 20352209), List(60928, 61957, 61267.1075, 400, 24506843), 0, null, null, 0, 1, 400, 0, true, 0, 0, 1793721549559, 1793721555276, 8, 1, null, List(0, 0), null, 13, 13, 1396, 0, null, null, 0)"


In [0]:
mesure("après")

après  numFiles=   1  sizeInBytes=20,352,209  requête=0.84 s


In [0]:
%sql
SELECT version, operation, operationMetrics
FROM (DESCRIBE HISTORY workspace.shopflow.payments_fragmented)
ORDER BY version DESC
LIMIT 1;

version,operation,operationMetrics
1,OPTIMIZE,"Map(numRemovedFiles -> 400, numRemovedBytes -> 24506843, p25FileSize -> 20352209, numDeletionVectorsRemoved -> 0, minFileSize -> 20352209, numAddedFiles -> 1, maxFileSize -> 20352209, p75FileSize -> 20352209, p50FileSize -> 20352209, numAddedBytes -> 20352209)"


In [0]:
%sql
-- Étape 4
VACUUM workspace.shopflow.payments_fragmented DRY RUN;

path


**Q10.** Avant : environ 400 fichiers. Après `OPTIMIZE` : quelques fichiers (voir `numFilesRemoved` / `numFilesAdded` dans `operationMetrics`). Les anciens fichiers restent sur le stockage car ils sont référencés par les **anciennes versions** de la table (time travel) : `OPTIMIZE` crée un commit qui retire les petits fichiers et ajoute les gros, sans rien effacer. `VACUUM ... DRY RUN` liste les fichiers qui seraient supprimés : aucun ne l'est encore, car ils ont été retirés il y a moins de 7 jours (rétention par défaut).

Le chronométrage est indicatif : sur serverless, la deuxième requête peut profiter d'un cache, et à SF1 l'écart reste faible.

---
## Partie 6 : restitution (éléments de réponse)

1. Pipeline attendu : volume `landing` (CSV paiements, CSV rétrofacturations, JSON navigation) → Auto Loader (`checkpoints`) → `bronze_payments`, `bronze_chargebacks`, `bronze_clickstream` → `silver_payments` (+ `silver_payments_quarantine`) → `gold_payment_features` (jointure avec `orders` et `customers`). Schéma de référence : `images/schemas/tp2-pipeline.svg` (à ne pas montrer avant la restitution).
2. Exemple de *time travel* : la comparaison de la partie 3 (fraudes connues de décembre 2024 avant et après le lot 2), ou la lecture de la table telle qu'un modèle l'aurait vue après le lot 1.
3. Exemples de features : nombre de pays de cartes distincts par IP sur 24 h (`test_de_cartes`), nombre de produits high-tech dans la commande (`compte_compromis`), nombre de pages vues avant l'achat depuis `bronze_clickstream` (un fraudeur va droit au paiement).

**_Bonus_** : la table `payments` du TP 1 contient la vérité terrain (`fraud_scenario`). En la joignant, on vérifie que chaque scénario est bien capté par la feature attendue, y compris les fraudes jamais rétrofacturées (`label = false`).

In [0]:
%sql
SELECT coalesce(p.fraud_scenario, 'légitime') AS scenario,
       COUNT(*) AS paiements,
       ROUND(AVG(CAST(g.label AS INT)), 3)                AS taux_label,
       ROUND(AVG(g.ip_nb_cards_1h), 2)                    AS cartes_par_ip,
       ROUND(AVG(g.ip_nb_tx_1h), 2)                       AS tx_par_ip_1h,
       ROUND(AVG(g.amount_ratio_30d), 2)                  AS ratio,
       ROUND(AVG(CAST(g.ship_country_mismatch AS INT)), 3) AS mismatch,
       ROUND(AVG(CASE WHEN g.account_age_days >= 0 THEN g.account_age_days END), 1) AS age_compte_moyen
FROM workspace.shopflow.gold_payment_features AS g
JOIN workspace.shopflow.payments AS p USING (payment_id)
GROUP BY 1
ORDER BY paiements DESC;

scenario,paiements,taux_label,cartes_par_ip,tx_par_ip_1h,ratio,mismatch,age_compte_moyen
légitime,1020035,0.0,1.0,1.02,1.0,0.0,515.8
compte_compromis,5819,0.896,1.0,1.01,1.42,0.984,449.5
test_de_cartes,5166,0.178,15.14,15.14,0.47,0.0,null
nouveau_compte_montant,960,0.895,1.0,1.0,1.0,0.979,0.5


- `ip_nb_cards_1h` et `ip_nb_tx_1h` très élevés pour le scénario `test_de_cartes` (15 à 40 cartes depuis une même IP en 30 min). Attention : la plupart de ces paiements sont **refusés**, donc jamais rétrofacturés : leur label observé est 0. C'est une limite du label à discuter.
- `ship_country_mismatch` vrai pour `compte_compromis` et `nouveau_compte_montant` (livraison vers un pays « à risque »).
- `amount_ratio_30d` élevé pour `compte_compromis` (achats high-tech sur un compte habituellement modeste).
- `account_age_days` faible (0 à 2 jours) pour `nouveau_compte_montant`.